# M03B Solution: Error Handling

This notebook contains the solution for the **Support Ticket Urgency Classifier** exercise from Module 03B.

**What's included:**
- The complete `ask_openai_v3` helper function (reference)
- The robust classifier solution pattern
- Production-ready error handling logic

---

## 🔧 Step 1: Setup

We need the final version of our helper function (`ask_openai_v3`) to build the solution.

In [ ]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv

import openai

load_dotenv(dotenv_path=Path("..") / ".env")
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

def ask_openai_v3(prompt, max_retries=3, min_length=1, max_length=10000):
    """Version 3: Adds response validation."""
    
    def _validate_response(text):
        """Validate response quality."""
        if not text or not text.strip():
            return False, "Empty response"
        if len(text) < min_length:
            return False, f"Response too short (min {min_length} chars)"
        if len(text) > max_length:
            return False, f"Response too long (max {max_length} chars)"
        return True, "Valid"
    
    RETRYABLE_ERRORS = (openai.RateLimitError, openai.APIConnectionError)
    
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                input=prompt
            )
            response_text = response.output_text.strip()
            
            is_valid, validation_msg = _validate_response(response_text)
            
            if is_valid:
                return {
                    "success": True,
                    "data": response_text,
                    "attempts": attempt + 1,
                    "validated": True
                }
            else:
                if attempt < max_retries - 1:
                    print(f"⚠️ Attempt {attempt + 1}: {validation_msg}. Retrying...")
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return {
                        "success": False,
                        "error": f"Validation failed: {validation_msg}",
                        "data": response_text
                    }
        
        except openai.AuthenticationError as e:
            return {
                "success": False,
                "error": "Authentication failed (auth)",
                "details": str(e)
            }
            
        except openai.BadRequestError as e:
            return {
                "success": False,
                "error": "Bad request (check model/prompt)",
                "details": str(e)
            }
        
        except RETRYABLE_ERRORS as e:
            error_type = "rate_limit" if isinstance(e, openai.RateLimitError) else "network"
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"⏳ Attempt {attempt + 1} failed ({error_type}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                return {
                    "success": False,
                    "error": f"Max retries exceeded ({error_type})",
                    "details": str(e)
                }
        
        except Exception as e:
            return {
                "success": False,
                "error": "Unexpected error",
                "details": str(e)
            }

print("✅ Ready!")

---

## 🎫 Solution: Support Ticket Urgency Classifier

This is one valid solution. Your implementation may differ in prompt wording, variable names, or structure — that's fine as long as it follows the core pattern.

**The pattern:**
1. **Define constraints:** Valid categories (`high`, `medium`, `low`).
2. **Define fallback:** Safe default (`medium`) if AI fails.
3. **Call the wrapper:** Use `ask_openai_v3` with retries.
4. **Validate content:** Ensure output matches valid categories.
5. **Handle failure:** Return fallback if validation fails or API errors occur.

In [ ]:
def robust_support_classifier(ticket_text):
    """Classify support ticket urgency."""
    
    # ✅ Explicit constraints and fallbacks
    VALID_CATEGORIES = ["high", "medium", "low"]
    FALLBACK = "medium"
    
    prompt = f"""Classify this support ticket urgency as: high, medium, or low.

Ticket: {ticket_text}

Return only the urgency level."""
    
    # Call the robust wrapper
    result = ask_openai_v3(prompt, max_retries=3)
    
    if result["success"]:
        urgency = result["data"].lower().strip()
        
        # ✅ Content validation
        if urgency in VALID_CATEGORIES:
            return {"success": True, "urgency": urgency, "source": "ai"}
        else:
            print(f"⚠️ Invalid category returned: {urgency}")
    
    # API failed or invalid response - use fallback
    return {"success": True, "urgency": FALLBACK, "source": "fallback"}


# --------------------------------------------------------------
# Test the solution
# --------------------------------------------------------------
test_tickets = [
    "System is completely down - all users affected!",
    "Minor typo on the settings page",
    "Can't login, have a demo in 1 hour",
]

print("🧪 SUPPORT TICKET CLASSIFIER")
print("="*60)
for ticket in test_tickets:
    result = robust_support_classifier(ticket)
    print(f"\nTicket: {ticket[:40]}...")
    print(f"   Urgency: {result['urgency']}")
    print(f"   Source: {result['source']}")
print("="*60)

---

## 🎯 Key Takeaways

**Wrapper Evolution:**
- **v0 (Raw):** Good for prototyping, dangerous for production.
- **v1 (Basic):** Prevents crashes, returns error info.
- **v2 (Retries):** Handles temporary network/rate glitches.
- **v3 (Production):** Adds validation and standardizes output.

**The "Robust Function" Pattern:**
- Always wrap the API call in a domain-specific function (like `robust_support_classifier`).
- Define **Valid Outputs** explicitly (enums, lists).
- Define a **Safe Fallback** (default value).
- Return a **Source** flag (AI vs. Fallback) for debugging.

---

### 📍 Next Step

**M03C: Prompt Templates & Management** — Learn how to organize and version control your prompts.

---